<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-HUYTRUONG/AIO-2026/blob/main/M3/ML-Base/Tree-based-Algorithms/demo_student_pass_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Bước 1. Nạp thư viện

In [1]:
import pandas as pd
import numpy as np
import gradio as gr
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.tree import DecisionTreeClassifier, plot_tree
from typing import Union, BinaryIO, TextIO, List
from dataclasses import dataclass

ModuleNotFoundError: No module named 'pandas'

## Bước 2. Nạp và mã hóa dữ liệu

In [ ]:
# %run ./drive/MyDrive/AIO/helpers/dataset_helper.ipynb

import os

GITHUB_RAW_URL = "https://raw.githubusercontent.com/AIVIETNAM-AIO-HUYTRUONG/AIO-2026/main/M3/ML-Base/Tree-based%20Algorithms/data/Student_Pass_Dataset.csv"
LOCAL_CSV_NAME = "Student_Pass_Dataset.csv"


def get_csv_path() -> str:
    """Trả về đường dẫn (hoặc URL) tới file CSV dữ liệu.

    Thứ tự ưu tiên (để notebook tự chạy được ở bất kỳ đâu, không phụ thuộc
    Google Drive cá nhân của tác giả):
      1. Tải trực tiếp từ GitHub (không cần mount Drive).
      2. Dùng file CSV đã có sẵn cùng thư mục với notebook.
      3. Nếu đang chạy trên Colab: cho phép upload file thủ công.
    """
    try:
        pd.read_csv(GITHUB_RAW_URL, nrows=1)
        print(f"Đã kết nối được tới dữ liệu trên GitHub:\n{GITHUB_RAW_URL}")
        return GITHUB_RAW_URL
    except Exception as e:
        print(f"Không tải được từ GitHub ({e}). Thử phương án khác...")

    if os.path.exists(LOCAL_CSV_NAME):
        print(f"Dùng file cục bộ: {LOCAL_CSV_NAME}")
        return LOCAL_CSV_NAME

    try:
        from google.colab import files
        print("Vui lòng chọn file CSV để upload...")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        print(f"Upload thành công: {filename}")
        return filename
    except ImportError:
        raise FileNotFoundError(
            "Không tìm thấy dữ liệu. Hãy đặt file 'Student_Pass_Dataset.csv' "
            "cùng thư mục với notebook, hoặc chạy trên Google Colab để upload thủ công."
        )


csv_path = get_csv_path()

In [ ]:
def encode_df(file_input: Union[str, BinaryIO, TextIO]) -> pd.DataFrame:
    """Tải và tự động mã hóa dữ liệu cho mọi tệp CSV."""
    df = pd.read_csv(file_input)
    binary_map = {'Có': 1, 'Không': 0}

    for col in df.select_dtypes(include=["object"]).columns:
      df[col] = df[col].astype(str).str.strip().map(binary_map)

    return df
df_encoded = encode_df(csv_path)
display(df_encoded)

### **Phân tích dữ liệu**

- **Điểm tốt nghiệp (numeric)**: Thuộc tính này cho biết khả năng học vấn của thí sinh, và vì là biến liên tục nên khi tách nhánh thì ta sẽ tìm **ngưỡng (threshold)** phù hợp.

- **Chứng chỉ IELTS (categorical)**: Cho biết thí sinh có chứng chỉ IELTS hay không — đây là biến rời rạc với hai giá trị **Có/Không**.

- **Cộng Điểm Dân Tộc (categorical)**: Cho biết thí sinh có được cộng điểm ưu tiên theo chính sách dân tộc hay không. Giá trị cũng là **Có/Không**.

- **Đậu Đại Học (label)**: Nhãn mục tiêu mà chúng ta muốn dự đoán. **Có** nếu thí sinh trúng tuyển, ngược lại là **Không**.

----
**Yêu cầu**: Xây dựng Classification Tree để dự đoán một sinh viên có trúng tuyển đại học không từ ba thuộc tính trên, theo 2 cách:
  1. Entropy & Information Gain.
  2. Gini Impurity & Gini Gain.

- Sau khi hoàn thiện, hãy thử dự đoán kết quả cho dữ liệu của một học sinh mới:

  **(Điểm Tốt Nghiệp = 21.0, Chứng chỉ IELTS = Không, Cộng Điểm Dân Tộc = Có)**

## Bước 3. Tính các chỉ số ban đầu (Đo độ "lộn xộn" của dữ liệu)

1. Tỷ lệ xuất hiện các nhãn: Tập dữ liệu ban đầu gồm 8 học sinh, trong đó có 3 học sinh mang nhãn "Có" (Đậu) và 5 học sinh mang nhãn "Không" (Không đậu).
    - Tỷ lệ học sinh "Có": $p_{\text{Có}} = \frac{3}{8} = 0.375$

    - Tỷ lệ học sinh "Không": $p_{\text{Không}} = \frac{5}{8} = 0.625$

2. Tính độ hỗn loạn (Entropy - $H$)
    - **Ý nghĩa**: Entropy đo mức độ "lộn xộn" hoặc "không chắc chắn" của dữ liệu. Nếu dữ liệu chia đều 50/50 thì Entropy đạt cực đại ($= 1$); nếu dữ liệu chỉ chứa đúng 1 loại nhãn thì Entropy $= 0$ (hoàn toàn thuần khiết).

    - Công thức:
    
      $$H(S_0) = -\sum p \log_2 p = -\left(p_{\text{Có}} \cdot \log_2 p_{\text{Có}} + p_{\text{Không}} \cdot \log_2 p_{\text{Không}}\right)$$

    - Thay số:
    
      $$H(S_0) = -\left(\frac{3}{8} \log_2 \frac{3}{8} + \frac{5}{8} \log_2 \frac{5}{8}\right) \approx 0.9544$$
    - Nhận xét: Kết quả $0.9544$ rất gần $1$, cho thấy tập dữ liệu hiện tại đang bị trộn lẫn khá nhiều.

3. Tính độ không thuần khiết (Gini Impurity - $G$)
    - **Ý nghĩa**: Gini là một chỉ số khác dùng để đo độ "vẩn đục" của tập dữ liệu. Gini càng nhỏ thì tập dữ liệu càng thuần khiết (Gini $= 0$ khi dữ liệu chỉ có 1 nhãn duy nhất); Gini càng lớn thì dữ liệu càng lẫn lộn giữa các nhãn.

    - Công thức:
    
      $$G(S_0) = 1 - \sum p^2 = 1 - \left(p_{\text{Có}}^2 + p_{\text{Không}}^2\right)$$

    - Thay số:
      
      $$G(S_0) = 1 - \left[\left(\frac{3}{8}\right)^2 + \left(\frac{5}{8}\right)^2\right] = 1 - \left(\frac{9}{64} + \frac{25}{64}\right) = 0.4688$$


In [ ]:
def calculate_entropy(probabilities:np.ndarray) -> float:
  """Tính Entropy: H = -sum(p * log2(p))."""
  valid_probs = probabilities[probabilities > 0]

  if len(valid_probs) == 0:
    return 0.0

  result = -np.sum(valid_probs * np.log2(valid_probs))
  return float(result)

def calculate_gini(probabilities:np.ndarray) -> float:
  """Tính Gini Impurity: G = 1 - sum(p^2)."""
  if len(probabilities) == 0:
    return 0.0

  result = 1 - np.sum(probabilities**2)
  return  float(result)

root_prods = df_encoded['Đậu_ĐH'].value_counts(normalize=True).sort_index().values

root_entropy = calculate_entropy(root_prods)
root_gini = calculate_gini(root_prods)

print(f"Root entropy của Đậu_ĐH = {root_entropy:.4f}")
print(f"Root gini của Đậu_ĐH = {root_gini:.4f}")

## Bước 4. Tính Gain cho từng thuộc tính

### 4.1. Điểm Tốt Nghiệp (numeric)

- Sắp xếp điểm số tăng dần: $12, 14.5, 16, 18, 20, 22, 24, 26$.
- Tính điểm trung bình giữa từng cặp điểm liên tiếp để tạo ra 7 mốc cắt (ngưỡng) thử nghiệm: $13.25, 15.25, 17, 19, 21, 23, 25$.

### 4.2. Minh họa cách tính thử với mốc **13.25**

- Chia 8 học sinh dựa trên câu hỏi: "**Điểm tốt nghiệp $\le$ 13.25**?"
  - Nhóm ĐÚNG (Có 1 học sinh - 12 điểm):
    - Kết quả: 0 Đậu, 1 Không.
    - Nhóm này đồng nhất tuyệt đối ($100\%$ không đậu) nên độ hỗn loạn bằng 0
      - $\text{Entropy} = 0 \rightarrow H(\text{True}) = 0$,
      - $\text{Gini} = 0\rightarrow G(\text{True}) = 0$.
  - Nhóm SAI (Có 7 học sinh còn lại):
    - Kết quả: 3 Đậu, 4 Không.
    - Nhóm này còn xáo trộn nhiều ($\text{Entropy} = 0.9852$, $\text{Gini} = 0.4898$).

      $$H(\text{False}) = -\frac{3}{7} \log_2 \left(\frac{3}{7}\right) - \frac{4}{7} \log_2 \left(\frac{4}{7}\right) = 0.9852$$


      $$G(\text{False}) = 1 - \left( \left(\frac{3}{7}\right)^2 + \left(\frac{4}{7}\right)^2 \right) = 0.4898$$

  - Mức độ giảm xáo trộn (Gain):
    - Information Gain (IG):
    
      \begin{aligned}
      IG(S_0, \text{Điểm tốt nghiệp} \le 13.25) &= H(S_0) - \left( \frac{|S_{\text{True}}|}{|S|} H(\text{True}) + \frac{|S_{\text{False}}|}{|S|} H(\text{False}) \right) \\[1.2ex]
      &= 0.9544 - \left( \frac{1}{8} \times 0 + \frac{7}{8} \times 0.9852 \right) \\[1.2ex]
      &= 0.9544 - 0.8620 \\[1.2ex]
      &= 0.0924
      \end{aligned}
    
    - Gini Gain (GG):

    \begin{aligned}
    GG(S_0, \text{Điểm tốt nghiệp} \le 13.25) &= G(S_0) - \left( \frac{|S_{\text{True}}|}{|S|} G(\text{True}) + \frac{|S_{\text{False}}|}{|S|} G(\text{False}) \right) \\[1.2ex]
    &= 0.4688 - \left( \frac{1}{8} \times 0 + \frac{7}{8} \times 0.4898 \right) \\[1.2ex]
    &= 0.4688 - 0.4286 \\[1.2ex]
    &= 0.0402
    \end{aligned}

### 4.3. Bảng so sánh hiệu quả của tất cả các mốc chia

| STT | Ngưỡng Điểm ($T$) | Điều Kiện Phân Nhánh | Information Gain ($IG$) | Gini Gain ($GG$) | Đánh Giá / Ghi Chú |
| :---: | :---: | :---: | :---: | :---: | :--- |
| 1 | **13.25** | Điểm $\le 13.25$ | 0.0924 | 0.0402 | Phân tách kém, hầu hết dữ liệu rơi vào 1 nhánh. |
| 2 | **15.25** | Điểm $\le 15.25$ | 0.2044 | 0.0938 | Hiệu quả thấp. |
| 3 | **17.00** | Điểm $\le 17.00$ | 0.3476 | 0.1688 | Mức độ phân tách trung bình. |
| 4 | **19.00** | **Điểm $\le 19.00$** | **0.5488** | **0.2812** | **TỐI ƯU NHẤT (Mức giảm hỗn loạn lớn nhất)** |
| 5 | **21.00** | Điểm $\le 21.00$ | 0.1589 | 0.1021 | Hiệu quả giảm mạnh. |
| 6 | **23.00** | Điểm $\le 23.00$ | 0.4669 | 0.2605 | Khá tốt, nhưng vẫn kém mốc 19.00. |
| 7 | **25.00** | Điểm $\le 25.00$ | 0.1992 | 0.1116 | Phân tách kém. |

---

#### Kết luận

1. **Ngưỡng được chọn:** Mốc **19.00** đạt cả **Information Gain ($0.5488$)** lẫn **Gini Gain ($0.2812$)** cao nhất.
2. **Ý nghĩa:** Chia tại mốc $19.00$ giúp tách biệt hoàn toàn nhóm chắc chắn rớt ($\le 19$ điểm có $100\%$ rớt) với nhóm học sinh có tỷ lệ đậu cao ($> 19$ điểm có $75\%$ đậu).

In [ ]:
@dataclass
class UniversalFeatureResult:
    """Lưu trữ kết quả đánh giá phân nhánh cho một thuộc tính bất kỳ."""
    feature_name: str
    feature_type: str            # 'Continuous (Threshold)' hoặc 'Categorical'
    best_split_condition: str    # Mô tả điều kiện chia (VD: '<= 19.0' hoặc '== values')
    weighted_entropy: float
    information_gain: float
    weighted_gini: float
    gini_gain: float


def evaluate_continuous_feature(
    df: pd.DataFrame, feature: str, target: str, root_entropy: float, root_gini: float
) -> UniversalFeatureResult:
    """Tìm điểm ngưỡng (threshold) tối ưu cho thuộc tính số."""
    total_count = len(df)
    sorted_values = np.sort(df[feature].unique())
    candidate_thresholds = (sorted_values[:-1] + sorted_values[1:]) / 2.0

    best_ig = -1.0
    best_res = None

    for t in candidate_thresholds:
        left_df = df[df[feature] <= t]
        right_df = df[df[feature] > t]

        w_left = len(left_df) / total_count
        w_right = len(right_df) / total_count

        h_left = calculate_entropy(left_df[target].value_counts(normalize=True).values)
        h_right = calculate_entropy(right_df[target].value_counts(normalize=True).values)
        g_left = calculate_gini(left_df[target].value_counts(normalize=True).values)
        g_right = calculate_gini(right_df[target].value_counts(normalize=True).values)

        w_entropy = w_left * h_left + w_right * h_right
        w_gini = w_left * g_left + w_right * g_right

        ig = root_entropy - w_entropy
        gg = root_gini - w_gini

        if ig > best_ig:
            best_ig = ig
            best_res = UniversalFeatureResult(
                feature_name=feature,
                feature_type="Continuous (Threshold)",
                best_split_condition=f"<= {t:.1f}",
                weighted_entropy=w_entropy,
                information_gain=ig,
                weighted_gini=w_gini,
                gini_gain=gg
            )

    return best_res


def evaluate_categorical_feature(
    df: pd.DataFrame, feature: str, target: str, root_entropy: float, root_gini: float
) -> UniversalFeatureResult:
    """Đánh giá thuộc tính danh mục theo các giá trị phân biệt."""
    total_count = len(df)
    unique_vals = df[feature].unique()

    w_entropy = 0.0
    w_gini = 0.0

    for val in unique_vals:
        sub_df = df[df[feature] == val]
        weight = len(sub_df) / total_count

        probs = sub_df[target].value_counts(normalize=True).values
        w_entropy += weight * calculate_entropy(probs)
        w_gini += weight * calculate_gini(probs)

    ig = root_entropy - w_entropy
    gg = root_gini - w_gini

    return UniversalFeatureResult(
        feature_name=feature,
        feature_type="Categorical",
        best_split_condition="Categories Split",
        weighted_entropy=w_entropy,
        information_gain=ig,
        weighted_gini=w_gini,
        gini_gain=gg
    )


def evaluate_all_features(
    df: pd.DataFrame,
    continuous_features: List[str],
    categorical_features: List[str],
    target: str
) -> pd.DataFrame:
    """Đánh giá toàn bộ thuộc tính và trả về bảng xếp hạng tối ưu."""
    root_probs = df[target].value_counts(normalize=True).values
    root_entropy = calculate_entropy(root_probs)
    root_gini = calculate_gini(root_probs)

    print("=========================================================================")
    print(f"BÁO CÁO PHÂN TÍCH NÚT GỐC (ROOT NODE) - {len(df)} mẫu")
    print(f" -> Root Entropy H(S0) = {root_entropy:.4f}")
    print(f" -> Root Gini G(S0)    = {root_gini:.4f}")
    print("=========================================================================\n")

    results: List[UniversalFeatureResult] = []

    # 1. Đánh giá tất cả thuộc tính số liên tục
    for feat in continuous_features:
        results.append(evaluate_continuous_feature(df, feat, target, root_entropy, root_gini))

    # 2. Đánh giá tất cả thuộc tính danh mục
    for feat in categorical_features:
        results.append(evaluate_categorical_feature(df, feat, target, root_entropy, root_gini))

    # 3. Tổng hợp bảng kết quả
    table_data = []
    for r in results:
        table_data.append({
            'Thuộc tính (Feature)': r.feature_name,
            'Loại dữ liệu': r.feature_type,
            'Điều kiện chia tốt nhất': r.best_split_condition,
            'Entropy Trọng Số': round(r.weighted_entropy, 4),
            'Information Gain (IG)': round(r.information_gain, 4),
            'Gini Trọng Số': round(r.weighted_gini, 4),
            'Gini Gain (GG)': round(r.gini_gain, 4)
        })

    result_df = pd.DataFrame(table_data)
    result_df = result_df.sort_values(by='Information Gain (IG)', ascending=False).reset_index(drop=True)
    return result_df

In [ ]:
# Khai báo phân loại thuộc tính
continuous_cols = ['Điểm_Tốt_Nghiệp']
categorical_cols = ['Chứng_chỉ_Ielts', 'Cộng_Điểm_Dân_Tộc']
target_col = 'Đậu_ĐH'

# Chạy đánh giá cho TẤT CẢ features
leaderboard_df = evaluate_all_features(
    df=df_encoded,
    continuous_features=continuous_cols,
    categorical_features=categorical_cols,
    target=target_col
)

print("BẢNG XẾP HẠNG TẤT CẢ CÁC THUỘC TÍNH ĐỂ CHỌN NÚT GỐC (ROOT NODE):")
display(leaderboard_df)

## Bước 5. Huấn luyện cây quyết định đầy đủ (Entropy & Gini) và trực quan hoá

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

features = ['Chứng_chỉ_Ielts', 'Điểm_Tốt_Nghiệp', 'Cộng_Điểm_Dân_Tộc']
feature_labels = ['Chứng chỉ IELTS', 'Điểm Tốt Nghiệp', 'Cộng Điểm Dân Tộc']

X = df_encoded[features]
y = df_encoded['Đậu_ĐH']

# Theo đúng yêu cầu đề bài: xây dựng Classification Tree theo CẢ 2 cách
clf_entropy = DecisionTreeClassifier(criterion='entropy', random_state=0)
clf_entropy.fit(X, y)

clf_gini = DecisionTreeClassifier(criterion='gini', random_state=0)
clf_gini.fit(X, y)

# clf mặc định (dùng ở Bước 6/7 bên dưới nếu không chỉ định cụ thể)
clf = clf_entropy

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

plot_tree(
    clf_entropy,
    feature_names=feature_labels,
    class_names=['Không', 'Có'],
    filled=True, rounded=True, proportion=True, impurity=False,
    fontsize=9, ax=axes[0]
)
axes[0].set_title('Decision Tree (criterion="entropy")')

plot_tree(
    clf_gini,
    feature_names=feature_labels,
    class_names=['Không', 'Có'],
    filled=True, rounded=True, proportion=True, impurity=False,
    fontsize=9, ax=axes[1]
)
axes[1].set_title('Decision Tree (criterion="gini")')

plt.tight_layout()
plt.show()

print("Lưu ý: Với tập dữ liệu nhỏ này, 2 cây Entropy và Gini cho ra CÙNG một cấu trúc rẽ nhánh "
      "(điều này không phải lúc nào cũng đúng với các bộ dữ liệu khác).")

## Bước 6. Kiểm thử với dữ liệu mới

In [ ]:
_mapping = {'Có': 1, 'Không': 0}

def encode_input(record):
    return [
        _mapping[record['Chứng_chỉ_Ielts']],
        record['Điểm_Tốt_Nghiệp'],
        _mapping[record['Cộng_Điểm_Dân_Tộc']]
    ]

def predict_pass(record, model=None):
    """Dự đoán Đậu/Không Đậu cho 1 học sinh mới.
    model=None -> dùng `clf` mặc định (cây Entropy)."""
    if model is None:
        model = clf
    x_list = encode_input(record)
    df_new = pd.DataFrame([x_list], columns=features)
    pred = model.predict(df_new)[0]
    return 'Có' if pred == 1 else 'Không'

# Đúng theo dữ liệu học sinh mới trong phần "Yêu cầu" ở trên
new_student = {
    'Điểm_Tốt_Nghiệp': 21.0,
    'Chứng_chỉ_Ielts': 'Không',
    'Cộng_Điểm_Dân_Tộc': 'Có'
}

print("Học sinh mới:", new_student)
print("Dự đoán (cây Entropy):", predict_pass(new_student, clf_entropy))
print("Dự đoán (cây Gini)   :", predict_pass(new_student, clf_gini))

## Bước 7. Demo với Gradio

In [ ]:
import gradio as gr

MODELS = {'Entropy': clf_entropy, 'Gini': clf_gini}

def predict_and_plot(diem, ielts, dan_toc, criterion):
    try:
        record = {
            'Điểm_Tốt_Nghiệp': float(diem),
            'Chứng_chỉ_Ielts': ielts,
            'Cộng_Điểm_Dân_Tộc': dan_toc
        }
        model = MODELS[criterion]
        result = predict_pass(record, model)
        fig, ax = plt.subplots(figsize=(8, 6))
        plot_tree(
            model,
            feature_names=feature_labels,
            class_names=['Không', 'Có'],
            filled=True, rounded=True, proportion=True, impurity=False,
            fontsize=8, ax=ax
        )
        ax.set_title(f'Decision Tree (criterion="{criterion.lower()}")')
        plt.tight_layout()
        return result, fig
    except Exception as e:
        return f"Lỗi nhập liệu: {e}", None

iface = gr.Interface(
    fn=predict_and_plot,
    inputs=[
        gr.Number(
            label="Điểm Tốt Nghiệp",
            value=None,
            placeholder="Ví dụ: 17.0"
        ),
        gr.Radio(
            choices=['Có', 'Không'],
            label="Chứng chỉ IELTS",
            value='Có'
        ),
        gr.Radio(
            choices=['Có', 'Không'],
            label="Cộng Điểm Dân Tộc",
            value='Không'
        ),
        gr.Radio(
            choices=['Entropy', 'Gini'],
            label="Tiêu chí xây cây (criterion)",
            value='Entropy'
        ),
    ],
    outputs=[
        gr.Textbox(label="Dự đoán Đậu Đại Học"),
        gr.Plot(label="Decision Tree")
    ],
    title="Demo Student Pass Predictor",
    description="""
**Hướng dẫn nhập liệu**
- **Điểm Tốt Nghiệp**: Nhập số điểm tốt nghiệp (float).
- **Chứng chỉ IELTS**: Chọn `Có` nếu bạn đã có chứng chỉ IELTS, ngược lại chọn `Không`.
- **Cộng Điểm Dân Tộc**: Chọn `Có` nếu bạn được cộng điểm dân tộc, ngược lại chọn `Không`.
- **Tiêu chí xây cây**: Chọn `Entropy` (Information Gain) hoặc `Gini` (Gini Gain) để so sánh 2 cây quyết định được huấn luyện theo 2 tiêu chí khác nhau.

Bấm **Run** để xem kết quả dự đoán và cây quyết định tương ứng.
"""
)

iface.launch()